In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os, sys

GOOGLE_DRIVE_PATH_POST_MYDRIVE = 'toxic_comment_detection'
GOOGLE_DRIVE_PATH = os.path.join('/content', 'drive', 'MyDrive', GOOGLE_DRIVE_PATH_POST_MYDRIVE)
print(os.listdir(GOOGLE_DRIVE_PATH))

os.chdir(GOOGLE_DRIVE_PATH)
sys.path.append(GOOGLE_DRIVE_PATH)

['.env', 'kaggle.json', '.pre-commit-config.yaml', '.python-version', '.gitignore', 'README.md', 'train.py', 'dataset_analysis.py', 'uv.lock', 'pyproject.toml', '.git', '.venv', 'notebooks', 'models', 'docs', 'dataset', '.ruff_cache', 'model_output']


In [1]:
# if running locally set GOOGLE PATH
import sys
if 'google.colab' in sys.modules:
  print(f'Running in google colab. Our path is `{GOOGLE_DRIVE_PATH}`')
else:
  GOOGLE_DRIVE_PATH = '.'
  print('Running locally.')

Running locally.


In [ ]:
!pip install transformers accelerate peft bitsandbytes datasets trl torch torchvision torchaudio

In [2]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import classification_report, f1_score


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

/home/shekhar/shekhar/toxic_comment_detection/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [ ]:
class Args:
    model_name = "meta-llama/Llama-3.2-1B-Instruct"
    batch_size = 32
    eval_batch_size = 32
    learning_rate = 2e-5
    weight_decay = 0.01
    epochs = 3
    grad_accum_steps = 1
    fp16 = True
    gradient_checkpoint = True
    optimizer = "adamw"
    warmup_ratio = 0.1
    max_length = 256
    val_ratio = 0.2 # 60-20-20
    test_ratio = 0.2
    output_dir = "./model_output"
    use_class_weights = True

args = Args()

In [6]:
import sys
import os


notebook_path = os.path.abspath("")  
llama_dir = os.path.dirname(notebook_path) 
models_dir = os.path.dirname(llama_dir)

sys.path.append(models_dir)
sys.path

['/home/shekhar/.local/share/uv/python/cpython-3.13.0-linux-x86_64-gnu/lib/python313.zip',
 '/home/shekhar/.local/share/uv/python/cpython-3.13.0-linux-x86_64-gnu/lib/python3.13',
 '/home/shekhar/.local/share/uv/python/cpython-3.13.0-linux-x86_64-gnu/lib/python3.13/lib-dynload',
 '',
 '/home/shekhar/shekhar/toxic_comment_detection/.venv/lib/python3.13/site-packages',
 '/home/shekhar/shekhar/toxic_comment_detection/.venv/lib/python3.13/site-packages/setuptools/_vendor',
 '/tmp/tmp2pwn0594',
 '/home/shekhar/shekhar/toxic_comment_detection']

In [14]:
dataset_dir = "/home/shekhar/shekhar/toxic_comment_detection/dataset/.cache/kagglehub/competitions/jigsaw-toxic-comment-classification-challenge/"
os.listdir(dataset_dir)

['test.csv.zip',
 'train.csv',
 'train.csv.zip',
 'test.csv',
 'test_labels.csv.zip',
 'sample_submission.csv.zip']

In [15]:
import pandas as pd
from datasets import Dataset, DatasetDict


train_df = pd.read_csv(f"{dataset_dir}train.csv")
test_df = pd.read_csv(f"{dataset_dir}test.csv")


train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)


data = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

full_train = data["train"]


test_val_split = full_train.train_test_split(test_size=args.test_ratio, seed=42)
train_temp = test_val_split["train"]
test_set = test_val_split["test"]

val_fraction = args.val_ratio / (1 - args.test_ratio)
train_val_split = train_temp.train_test_split(test_size=val_fraction, seed=42)
train_set = train_val_split["train"]
val_set = train_val_split["test"]

print(f"Split sizes: {len(train_set)} train, {len(val_set)} validation, {len(test_set)} test")


Split sizes: 95742 train, 31914 validation, 31915 test


In [11]:
# Cell 4: Tokenizer and data encoding
tokenizer = AutoTokenizer.from_pretrained(args.model_name, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "right"

def preprocess_batch(batch):
    encodings = tokenizer(batch["comment_text"], truncation=True, max_length=args.max_length, padding="longest")
    labels = np.array([batch[label] for label in ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]]).T
    encodings["labels"] = labels.tolist()
    return encodings

cols_to_remove = ["id", "comment_text", "toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
train_enc = train_set.map(preprocess_batch, batched=True, remove_columns=cols_to_remove)
val_enc = val_set.map(preprocess_batch, batched=True, remove_columns=cols_to_remove)
test_enc = test_set.map(preprocess_batch, batched=True, remove_columns=cols_to_remove)


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Map:   0%|          | 0/95742 [00:00<?, ? examples/s]

Map:   0%|          | 0/31914 [00:00<?, ? examples/s]

Map:   0%|          | 0/31915 [00:00<?, ? examples/s]

In [12]:
model = AutoModelForSequenceClassification.from_pretrained(
    args.model_name,
    num_labels=6,
    problem_type="multi_label_classification"
).to(device)

model.config.pad_token_id = tokenizer.pad_token_id

if args.gradient_checkpoint:
    model.gradient_checkpointing_enable()

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

pos_weight = None
if args.use_class_weights:
    train_labels = np.array(train_enc["labels"])
    pos_counts = train_labels.sum(axis=0)
    neg_counts = len(train_enc) - pos_counts
    pos_weight_values = neg_counts / np.clip(pos_counts, 1, None)
    pos_weight = torch.tensor(pos_weight_values, dtype=torch.float).to(device)

class MultilabelTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False,num_items_in_batch=None):
        labels = inputs.pop("labels")  # Remove labels from inputs dictionary
        outputs = model(**inputs)      # Forward pass without labels
        logits = outputs.get("logits")
        # Ensure labels are float32
        labels = labels.to(logits.dtype)
        loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


trainer = MultilabelTrainer(
    model=model,
    args=TrainingArguments(
        output_dir=args.output_dir,
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.eval_batch_size,
        gradient_accumulation_steps=args.grad_accum_steps,
        learning_rate=args.learning_rate,
        weight_decay=args.weight_decay,
        fp16=args.fp16,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_strategy="epoch",
        report_to="none",
        warmup_ratio=args.warmup_ratio,
        optim="adamw_torch" if args.optimizer == "adamw" else "adafactor"
    ),
    train_dataset=train_enc,
    eval_dataset=val_enc,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=lambda eval_pred: {
        "f1_macro": f1_score(eval_pred.label_ids, (torch.sigmoid(torch.tensor(eval_pred.predictions)) >= 0.5).int(), average="macro"),
        "f1_micro": f1_score(eval_pred.label_ids, (torch.sigmoid(torch.tensor(eval_pred.predictions)) >= 0.5).int(), average="micro")
    }
)


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Llama-3.2-1B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-12-490a9f108c49>:34: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `MultilabelTrainer.__init__`. Use `processing_class` instead.
  trainer = MultilabelTrainer(


In [14]:
trainer.train()



`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss,Model Preparation Time,F1 Macro,F1 Micro
1,0.558600,0.501700,0.002800,0.596256,0.716304
2,0.333200,0.429442,0.002800,0.596622,0.700984
3,0.161800,1.050213,0.002800,0.670013,0.783282


TrainOutput(global_step=8976, training_loss=0.3511660876758596, metrics={'train_runtime': 6651.2149, 'train_samples_per_second': 43.184, 'train_steps_per_second': 1.35, 'total_flos': 4.293371821031424e+17, 'train_loss': 0.3511660876758596, 'epoch': 3.0})

In [15]:

predictions = trainer.predict(test_enc)
y_true = predictions.label_ids
y_prob = torch.sigmoid(torch.tensor(predictions.predictions)).numpy()
y_pred = (y_prob >= 0.5).astype(int)

target_names = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
print(classification_report(y_true, y_pred, target_names=target_names, digits=4))


               precision    recall  f1-score   support

        toxic     0.7676    0.8764    0.8184      3018
 severe_toxic     0.3983    0.6288    0.4877       299
      obscene     0.7449    0.9075    0.8182      1676
       threat     0.4862    0.6092    0.5408        87
       insult     0.6920    0.8529    0.7641      1570
identity_hate     0.4927    0.6306    0.5532       268

    micro avg     0.7086    0.8550    0.7750      6918
    macro avg     0.5969    0.7509    0.6637      6918
 weighted avg     0.7148    0.8550    0.7779      6918
  samples avg     0.0701    0.0806    0.0723      6918



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
